# PlantVillage GPU Training (Google Colab)

Train `BaselineCNN`, `ResNet18`, or `EfficientNet-B0` on PlantVillage with GPU acceleration and export safe `state_dict` checkpoints.

In [ ]:
!git clone https://github.com/GaneshArwan/image_classification.git
%cd image_classification
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Load PlantVillage Dataset
Option A downloads the public PlantVillage dataset via Hugging Face `datasets` into `ImageFolder` format (`data/plantvillage/<class>/`) with no API key required.

In [ ]:
!pip install -q datasets
from pathlib import Path
from datasets import load_dataset

out_root = Path('data/plantvillage')
out_root.mkdir(parents=True, exist_ok=True)
ds = load_dataset('gvlassis/California_PlantVillage', split='train') if False else load_dataset('BrandonFors/Plant-Diseases-PlantVillage-Dataset', split='train')
label_names = ds.features['label'].names
for idx, row in enumerate(ds):
    cls_dir = out_root / label_names[row['label']].replace(' ', '_')
    cls_dir.mkdir(parents=True, exist_ok=True)
    row['image'].convert('RGB').save(cls_dir / f'{idx:06d}.jpg', format='JPEG')
print('Prepared classes:', len(list(out_root.iterdir())))

In [ ]:
# Train and compare Baseline CNN vs ResNet18 (with and without augmentation)
!python -m src.train --data-dir data/plantvillage --compare-all --epochs 10 --batch-size 32 --image-size 224 --output-dir artifacts/plantvillage_run
!cat artifacts/plantvillage_run/comparison_summary.json

In [ ]:
# Optional: Train EfficientNet-B0 with pretrained weights
!python -m src.train --data-dir data/plantvillage --architecture efficientnet_b0 --epochs 12 --batch-size 32 --image-size 224 --output-dir artifacts/plantvillage_run
!zip -r -q plantvillage_artifacts.zip artifacts/plantvillage_run
from google.colab import files
files.download('plantvillage_artifacts.zip')